In [1]:
from pyrocko import util, model, io, trace, moment_tensor, gmtpy,orthodrome
import pyrocko.moment_tensor as pmt
from pyrocko import orthodrome as od
from pyrocko.guts import load

# from seiscloud import plot as scp
# from seiscloud import cluster as scc
import numpy as np
import os
# import shutil
import matplotlib.pyplot as plt

import re
from pathlib import Path
from datetime import datetime

import yaml

# CLASS TO LOAD .YAML FILES AND READ PARAMENTER RESULTS
class IgnoreTagsLoader(yaml.SafeLoader):
    pass

def ignore_unknown(loader, tag_suffix, node):
    if isinstance(node, yaml.MappingNode):
        return loader.construct_mapping(node)
    elif isinstance(node, yaml.SequenceNode):
        return loader.construct_sequence(node)
    else:
        return loader.construct_scalar(node)

IgnoreTagsLoader.add_multi_constructor('!', ignore_unknown)

In [2]:
workdir='../' 
reportdir=os.path.join(workdir,'report')                                

catdir=os.path.join(workdir,'CAT')
catname=os.path.join(catdir,'catalogue_flegrei_VLP_old.pf')                 # CHANGE
refevents=model.load_events(catname)
mttargets = [ev for ev in refevents]

badmtsols = ['']    # exclude some events
print(f'Catalogue: {catname}')
print('All events in catalogue:', len(mttargets))
goodmttargets = [ev for ev in mttargets if ev.name not in badmtsols]
print('Good events in catalogue:', len(goodmttargets))

# Insert inversions names
inversions= ['cmt_LP_oscill_']                                    # CHANGE

# Insert source prefixes 
source_prefixes = ['']                                             # CHANGE
par_names = ['time','north_shift','east_shift',
              'depth','magnitude',
              'rmnn','rmee','rmdd','rmne','rmnd','rmed',
              'duration','frequency',
              'strike1','dip1','rake1']

print(f'\nInversions selected: {inversions}')
print(f'Sources extracted: {source_prefixes}')

ev_stats={}

# new catalogue name
new_catalogue_name = 'catalogue_flegrei_composite_MT_LF_std'

Catalogue: ../CAT/catalogue_flegrei_VLP_old.pf
All events in catalogue: 14
Good events in catalogue: 14

Inversions selected: ['cmt_LP_oscill_']
Sources extracted: ['']


In [3]:
reportdir=os.path.join(workdir,'report') # pre-defined
#reportdir = '/Users/giaco/UNI/PhD_CODE/GIT/CAMPI_FLEGREI_moment_tensor/report'

counter_total_reports = 0
for ev in goodmttargets:
    counter = 0
    for vrs in inversions: # main report
        targetdir = os.path.join(reportdir, ev.name, vrs + ev.name)
        #if not os.path.isdir(targetdir):
            #print(ev.name, 'missing report dir', targetdir)
        if os.path.isdir(targetdir):
            counter += 1
            fname = os.path.join(targetdir, 'stats.yaml')     # results
            if os.path.isfile(fname):
                # Read file
                with open(fname, 'r') as f:
                    data = yaml.load(f, Loader=IgnoreTagsLoader)    # unsing loader class
                
                # acces 'paramenter_stats_list'
                parameter_stats = data['parameter_stats_list']
                #print(parameter_stats)
                for pref in source_prefixes:
                    # combined grond code
                    #ev_stats[ev.name]={pref +'.'+ par :0 for par in par_names}
                    ev_stats[ev.name]={par :0 for par in par_names}
                for p in parameter_stats:
                    if p['name'] in list(ev_stats[ev.name].keys()):
                        #CHANGE: choose values to extract
                        ev_stats[ev.name][p['name']]= {'best' : float( f"{p['best']:.8g}" ),   
                                                        'percentile16' : float( f"{p['percentile16']:.8g}" ),
                                                        'percentile84' : float( f"{p['percentile84']:.8g}" ),
                                                        'mean' : float( f"{p['mean']:.8g}" ),
                                                        'std' : float( f"{p['std']:.8g}" )
                                                        }
                        # print values
                        #print(p['name'], p['best'], p['percentile16'], p['percentile84'])

    if counter == 0:
        print(f'WARNING: {ev.name} does not have any report directories!')
    elif counter > 1:
        print(f'WARNING: {ev.name} has MULTIPLE report directories!')
    else:
        print(f'NICE! {ev.name} has report directory: {targetdir}')
        counter_total_reports += 1

print(f'\nTotal events with report directories: {counter_total_reports} out of {len(goodmttargets)}')

NICE! flegrei_2018_09_18_21_36_41 has report directory: ../report/flegrei_2018_09_18_21_36_41/cmt_LP_oscill_flegrei_2018_09_18_21_36_41
NICE! flegrei_2023_06_11_06_44_25 has report directory: ../report/flegrei_2023_06_11_06_44_25/cmt_LP_oscill_flegrei_2023_06_11_06_44_25
NICE! flegrei_2023_09_07_17_45_28 has report directory: ../report/flegrei_2023_09_07_17_45_28/cmt_LP_oscill_flegrei_2023_09_07_17_45_28
NICE! flegrei_2023_09_26_07_10_29 has report directory: ../report/flegrei_2023_09_26_07_10_29/cmt_LP_oscill_flegrei_2023_09_26_07_10_29
NICE! flegrei_2023_10_02_20_08_26 has report directory: ../report/flegrei_2023_10_02_20_08_26/cmt_LP_oscill_flegrei_2023_10_02_20_08_26
NICE! flegrei_2024_04_27_03_44_56 has report directory: ../report/flegrei_2024_04_27_03_44_56/cmt_LP_oscill_flegrei_2024_04_27_03_44_56
NICE! flegrei_2024_05_22_06_28_00 has report directory: ../report/flegrei_2024_05_22_06_28_00/cmt_LP_oscill_flegrei_2024_05_22_06_28_00
NICE! flegrei_2024_06_08_01_52_04 has report dir

In [6]:
# Extracting ellipses parameters and saving to file
ellipses_filename='uncertainty_ellipses_old' #CHANGE

#ellipses= ['# event_name lat_min lon_min lat_max lon_max'] # header for file
ellipses= ['#event_name lat_event_VT lon_event_VT east_shift_16 east_shift_84 nord_shift_16 nord_shift_84'] # header for file

for ev in goodmttargets:
    name = ev.name
    if name not in ev_stats:
        print(f'WARNING: {name} does not have extracted stats!')
        continue

    lat= ev.lat
    lon= ev.lon

    n16=ev_stats[name]['north_shift']['percentile16']
    n84=ev_stats[name]['north_shift']['percentile84']
    nbest=ev_stats[name]['north_shift']['best']
    if n16 < nbest < n84:
        print(f'{name} - north_shift: best={nbest}, 16th={n16}, 84th={n84} - OK')
    else:
        print(f'{name} - north_shift: best={nbest}, 16th={n16}, 84th={n84} - WARNING: best is outside the 16th-84th range!')
    e16=ev_stats[name]['east_shift']['percentile16']
    e84=ev_stats[name]['east_shift']['percentile84']
    ebest=ev_stats[name]['east_shift']['best']
    if e16 < ebest < e84:
        print(f'{name} - east_shift: best={ebest}, 16th={e16}, 84th={e84} - OK')
    else:
        print(f'{name} - east_shift: best={ebest}, 16th={e16}, 84th={e84} - WARNING: best is outside the 16th-84th range!')

    #nlat, nlon = od.ne_to_latlon(lat, lon,
    #                            np.array([n16,n84]),
    #                            np.array([e16,n84]))
    #ellipses.append(f'{name} {nlat[0]} {nlat[1]} {nlon[0]} {nlon[1]}')
    ellipses.append(f'{name} {lat} {lon} {e16} {e84} {n16} {n84}')

# Save ellipses to file
ellipses_file = os.path.join(catdir, ellipses_filename + '.txt')
with open(ellipses_file, 'w') as f:
    for line in ellipses:
        f.write(line + '\n')
print(f'\nEllipses saved to: {ellipses_file}')

flegrei_2018_09_18_21_36_41 - north_shift: best=-1279.6032, 16th=-1275.2755, 84th=1668.1789 - WARNING: best is outside the 16th-84th range!
flegrei_2018_09_18_21_36_41 - east_shift: best=-2856.9726, 16th=-4509.6838, 84th=-1124.4074 - OK
flegrei_2023_06_11_06_44_25 - north_shift: best=-749.48542, 16th=-1834.6089, 84th=-427.00102 - OK
flegrei_2023_06_11_06_44_25 - east_shift: best=427.51475, 16th=210.54077, 84th=2025.8504 - OK
flegrei_2023_09_07_17_45_28 - north_shift: best=-335.79144, 16th=-1294.9386, 84th=254.18912 - OK
flegrei_2023_09_07_17_45_28 - east_shift: best=-2354.5562, 16th=-2989.0735, 84th=-1716.1778 - OK
flegrei_2023_09_26_07_10_29 - north_shift: best=1155.4153, 16th=785.59762, 84th=2019.7711 - OK
flegrei_2023_09_26_07_10_29 - east_shift: best=692.79626, 16th=-662.26808, 84th=1494.7325 - OK
flegrei_2023_10_02_20_08_26 - north_shift: best=-1210.3048, 16th=-1433.8721, 84th=88.507207 - OK
flegrei_2023_10_02_20_08_26 - east_shift: best=-2060.982, 16th=-3128.9141, 84th=-1118.9284

In [ ]:
# mean values and std
depth=[]
std=[]
mag=[]
freq=[]
freq_std=[]
duration=[]
duration_std=[]
for ev_name in ev_stats:
    d = ev_stats[ev_name]['vlp.depth']['best']
    depth.append(d)
    s = ev_stats[ev_name]['vlp.depth']['std']
    std.append(s)

    m = ev_stats[ev_name]['vlp.magnitude']['best']
    mag.append(m)

    f = ev_stats[ev_name]['vlp.frequency']['best']
    freq.append(f)
    std_f = ev_stats[ev_name]['vlp.frequency']['std']
    freq_std.append(std_f)

    dur = ev_stats[ev_name]['vlp.duration']['best']
    duration.append(dur)
    std_dur = ev_stats[ev_name]['vlp.duration']['std']
    duration_std.append(std_dur)


m_depth=np.mean(depth)
m_std= np.mean(std)

max_mag=np.max(mag)

m_freq=np.mean(freq)
m_freq_std=np.mean(freq_std)

m_duration=np.mean(duration)
m_duration_std=np.mean(duration_std)

print('VLP best depth:\t\t\t',m_depth,'+-',m_std)
print('Max best magnitude:\t\t',max_mag)
print('VLP best frequency:\t\t',m_freq,'+-',m_freq_std)
print('VLP best duration:\t\t',m_duration,'+-',m_duration_std)

In [ ]:
relocate=True             #CHANGE
update_time=False          #CHANGE
solution = 'best'          #CHANGE w/ 'mean'

events_results = []
for ev in goodmttargets:
    if ev.name in list(ev_stats.keys()) :
        for source_name in source_prefixes:
        # create events objects for sub-problems
            m0=pmt.magnitude_to_moment(ev_stats[ev.name][source_name +'.'+'magnitude'][solution])
            tmp_mt = pmt.MomentTensor(mnn=ev_stats[ev.name][source_name +'.'+'rmnn'][solution] * m0,
                                    mee=ev_stats[ev.name][source_name +'.'+'rmee'][solution] * m0,
                                    mdd=ev_stats[ev.name][source_name +'.'+'rmdd'][solution] * m0,
                                    mne=ev_stats[ev.name][source_name +'.'+'rmne'][solution] * m0,
                                    mnd=ev_stats[ev.name][source_name +'.'+'rmnd'][solution] * m0,
                                    med=ev_stats[ev.name][source_name +'.'+'rmed'][solution] * m0,
                                    moment=m0)

            if relocate:    # re-locate
                nlat, nlon = od.ne_to_latlon(ev.lat, ev.lon,
                                np.array([ev_stats[ev.name][source_name +'.'+'north_shift'][solution]]),
                                np.array([ev_stats[ev.name][source_name +'.'+'east_shift'][solution]]))
                lat, lon = nlat[0], nlon[0]
            else:
                lat, lon = ev.lat , ev.lon

            if update_time:
                time = ev.time + ev_stats[ev.name][source_name +'.'+'time'][solution]
            else:
                time = ev.time

            tmp_ev=model.Event(lat=lat, 
                                lon=lon, 
                                time=time,
                                name=ev.name, 
                                depth=ev_stats[ev.name][source_name +'.'+'depth'][solution], 
                                elevation=None, 
                                magnitude=ev_stats[ev.name][source_name +'.'+'magnitude'][solution], 
                                moment_tensor=tmp_mt, 
                                duration=ev_stats[ev.name][source_name +'.'+'duration'][solution], 
                                tags=[f"sub-problem:{source_name}",f"frequency:{str(ev_stats[ev.name][source_name +'.'+'frequency'][solution])}" ] 
                                )            
            # Add event to dict events
            events_results.append(tmp_ev)
    else:
        print('WARNING: missing report dir for event:',ev.name)

events_results.sort(key=lambda x: x.time, reverse=False)
print('\n TOTAL EVENTS FOR CONFIG',inversions,' AND SOURCES',source_prefixes, ':', len(events_results))

if relocate:
    new_catalogue_path=os.path.join(catdir,new_catalogue_name+'_reloc_'+solution+'.pf')   
    model.dump_events(events_results, new_catalogue_path)
    print(f'\nCatalogue with relocated events saved to: {new_catalogue_path}')
else:
    new_catalogue_path=os.path.join(catdir,new_catalogue_name+'_'+solution+'.pf')   
    model.dump_events(events_results, new_catalogue_path)
    print(f'\nCatalogue with original locations saved to: {new_catalogue_path}')